# BÀI THỰC HÀNH 2: TẬP DỮ LIỆU VINAFOOD21 (BÀI 2, 3, 4)
Notebook gộp gồm 3 mô hình trên tập dữ liệu **VinaFood21** (21 lớp):
1. **Bài 2:** Mô hình **GoogLeNet** (Inception Blocks, Conv đầu `padding=3`, MaxPool `ceil_mode=True`)
2. **Bài 3:** Mô hình **ResNet-18** (Residual Connection, MaxPool `kernel=3, stride=2, padding=0` giữa các block)
3. **Bài 4:** Fine-tune **ResNet-50** pretrained từ HuggingFace (`microsoft/resnet-50`)

## 1. Chuẩn bị Môi trường & Tải Dữ liệu VinaFood21

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import ResNetForImageClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

# Tạo các thư mục lưu checkpoint
os.makedirs("./checkpoints/assignment-2", exist_ok=True)
os.makedirs("./checkpoints/assignment-3", exist_ok=True)
os.makedirs("./checkpoints/assignment-4", exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sử dụng thiết bị: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Sử dụng thiết bị: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [ ]:
# Xác định đường dẫn tập train và test
if os.path.exists("./VinaFood21/VinaFood21/train"):
    TRAIN_DIR = "./VinaFood21/VinaFood21/train"
    TEST_DIR  = "./VinaFood21/VinaFood21/test"
elif os.path.exists("./VinaFood21/train"):
    TRAIN_DIR = "./VinaFood21/train"
    TEST_DIR  = "./VinaFood21/test"
else:
    TRAIN_DIR = r"D:/UIT Files/DS201/Thực hành/Buổi 2/VinaFood21/VinaFood21/train"
    TEST_DIR  = r"D:/UIT Files/DS201/Thực hành/Buổi 2/VinaFood21/VinaFood21/test"

print(f"Train path: {TRAIN_DIR}")
print(f"Test path : {TEST_DIR}")

Train path: D:/UIT Files/DS201/Thực hành/Buổi 2/VinaFood21/VinaFood21/train
Test path : D:/UIT Files/DS201/Thực hành/Buổi 2/VinaFood21/VinaFood21/test


In [3]:
# Data Preprocessing & DataLoaders (dùng chung cho cả 3 bài)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_set = datasets.ImageFolder(TRAIN_DIR, transform=transform_train)
test_set  = datasets.ImageFolder(TEST_DIR,  transform=transform_test)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

NUM_CLASSES = len(train_set.classes)
print(f"Số lượng lớp ({NUM_CLASSES}): {train_set.classes}")
print(f"Tập huấn luyện (Train): {len(train_set):,} mẫu")
print(f"Tập kiểm tra   (Test) : {len(test_set):,} mẫu")

Số lượng lớp (21): ['banh-can', 'banh-hoi', 'banh-mi-chao', 'banh-tet', 'banh-trang-tron', 'banh-u', 'banh-uot', 'bap-nuong', 'bo-kho', 'bo-la-lot', 'bot-chien', 'ca-ri', 'canh-kho-qua', 'canh-khoai-mo', 'ga-nuong', 'goi-ga', 'ha-cao', 'hoanh-thanh-nuoc', 'pha-lau', 'tau-hu', 'thit-kho-trung']
Tập huấn luyện (Train): 10,044 mẫu
Tập kiểm tra   (Test) : 6,682 mẫu


In [4]:
# Hàm đánh giá chung cho 4 độ đo (Accuracy, Precision, Recall, F1-macro)
def evaluate(model, loader, device, is_hf=False):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            if is_hf:
                logits = model(pixel_values=images).logits
            else:
                logits = model(images)
            out = logits.argmax(dim=-1).cpu().numpy()
            preds.extend(out)
            trues.extend(labels.numpy())
    return {
        "accuracy":  accuracy_score(trues, preds),
        "precision": precision_score(trues, preds, average="macro", zero_division=0),
        "recall":    recall_score   (trues, preds, average="macro", zero_division=0),
        "f1":        f1_score       (trues, preds, average="macro", zero_division=0),
    }

---
## 2. Bài 2: Mô hình GoogLeNet trên VinaFood21
- Kiến trúc gồm các khối **Inception Blocks** với 4 nhánh song song
- Lớp Convolution đầu tiên có `padding = 3`
- Tất cả các lớp Max Pooling đều bật `ceil_mode = True`
- Huấn luyện với **Adam optimizer**, lưu lại model có **F1-macro** tốt nhất

In [5]:
class InceptionBlock(nn.Module):
    def __init__(self, in_ch, b1_out, b2_r, b2_out, b3_r, b3_out, b4_out):
        super().__init__()
        # Nhánh 1: 1x1 conv
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, b1_out, 1), nn.BatchNorm2d(b1_out), nn.ReLU(inplace=True))
        # Nhánh 2: 1x1 conv -> 3x3 conv
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_ch,  b2_r,   1), nn.BatchNorm2d(b2_r),   nn.ReLU(inplace=True),
            nn.Conv2d(b2_r,   b2_out, 3, padding=1), nn.BatchNorm2d(b2_out), nn.ReLU(inplace=True))
        # Nhánh 3: 1x1 conv -> 5x5 conv
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch,  b3_r,   1), nn.BatchNorm2d(b3_r),   nn.ReLU(inplace=True),
            nn.Conv2d(b3_r,   b3_out, 5, padding=2), nn.BatchNorm2d(b3_out), nn.ReLU(inplace=True))
        # Nhánh 4: 3x3 MaxPool -> 1x1 conv
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1, ceil_mode=True),
            nn.Conv2d(in_ch, b4_out, 1), nn.BatchNorm2d(b4_out), nn.ReLU(inplace=True))

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x),
                          self.branch3(x), self.branch4(x)], dim=1)

class GoogLeNet(nn.Module):
    def __init__(self, num_classes=21):
        super().__init__()
        # Stem: Conv đầu tiên có padding=3, MaxPool có ceil_mode=True
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, ceil_mode=True),
            nn.Conv2d(64,  64,  kernel_size=1),
            nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64,  192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, ceil_mode=True),
        )
        # Inception Group 3
        self.block3a = InceptionBlock(192, 64,  96,  128, 16,  32,  32)
        self.block3b = InceptionBlock(256, 128, 128, 192, 32,  96,  64)
        self.pool3   = nn.MaxPool2d(3, stride=2, ceil_mode=True)
        # Inception Group 4
        self.block4a = InceptionBlock(480, 192, 96,  208, 16,  48,  64)
        self.block4b = InceptionBlock(512, 160, 112, 224, 24,  64,  64)
        self.block4c = InceptionBlock(512, 128, 128, 256, 24,  64,  64)
        self.block4d = InceptionBlock(512, 112, 144, 288, 32,  64,  64)
        self.block4e = InceptionBlock(528, 256, 160, 320, 32,  128, 128)
        self.pool4   = nn.MaxPool2d(3, stride=2, ceil_mode=True)
        # Inception Group 5
        self.block5a = InceptionBlock(832, 256, 160, 320, 32,  128, 128)
        self.block5b = InceptionBlock(832, 384, 192, 384, 48,  128, 128)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc      = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.pool3(self.block3b(self.block3a(x)))
        x = self.pool4(self.block4e(self.block4d(self.block4c(self.block4b(self.block4a(x))))))
        x = self.block5b(self.block5a(x))
        x = self.dropout(self.avgpool(x).view(x.size(0), -1))
        return self.fc(x)

model_googlenet = GoogLeNet(num_classes=NUM_CLASSES).to(device)
print(f"GoogLeNet Parameters: {sum(p.numel() for p in model_googlenet.parameters()):,}")

GoogLeNet Parameters: 5,603,285


In [6]:
# Huấn luyện GoogLeNet
loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_googlenet.parameters(), lr=1e-3)

EPOCHS  = 10
best_f1_googlenet = 0.0

for epoch in range(1, EPOCHS + 1):
    model_googlenet.train()
    losses = []
    with tqdm(train_loader, desc=f"[GoogLeNet Epoch {epoch:02d}/{EPOCHS}]", ncols=90) as pb:
        for images, labels in pb:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model_googlenet(images), labels)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            pb.set_postfix(loss=f"{np.mean(losses):.4f}")

    s = evaluate(model_googlenet, test_loader, device)
    print(f"  Test: Acc={s['accuracy']:.4f} | Prec={s['precision']:.4f} | Rec={s['recall']:.4f} | F1={s['f1']:.4f}")

    if s["f1"] > best_f1_googlenet:
        best_f1_googlenet = s["f1"]
        torch.save(model_googlenet.state_dict(), "./checkpoints/assignment-2/best_model.pth")

print(f"\nGoogLeNet - Best Test F1: {best_f1_googlenet:.4f}")

Epoch [1/5] (52.7s) | Loss: 2.7999 | Acc: 0.1629 | Prec: 0.1937 | Rec: 0.1629 | F1: 0.0997
Epoch [2/5] (49.5s) | Loss: 2.5034 | Acc: 0.2105 | Prec: 0.1845 | Rec: 0.2105 | F1: 0.1689
Epoch [3/5] (48.7s) | Loss: 2.3651 | Acc: 0.2190 | Prec: 0.2009 | Rec: 0.2190 | F1: 0.1677
Epoch [4/5] (48.9s) | Loss: 2.3164 | Acc: 0.2143 | Prec: 0.2563 | Rec: 0.2143 | F1: 0.1931
Epoch [5/5] (51.8s) | Loss: 2.2101 | Acc: 0.2695 | Prec: 0.3029 | Rec: 0.2695 | F1: 0.2512

GoogLeNet - Best Test F1: 0.2512


In [7]:
# Đánh giá mô hình GoogLeNet tốt nhất
model_googlenet.load_state_dict(torch.load("./checkpoints/assignment-2/best_model.pth", map_location=device))
res_googlenet = evaluate(model_googlenet, test_loader, device)

print("=" * 55)
print("     KẾT QUẢ ĐÁNH GIÁ — GoogLeNet trên VinaFood21")
print("=" * 55)
for k, v in res_googlenet.items():
    print(f"  {k.capitalize():<12}: {v:.4f}")
print("=" * 55)

     KẾT QUẢ ĐÁNH GIÁ — GoogLeNet trên VinaFood21
  Accuracy    : 0.2695
  Precision   : 0.3029
  Recall      : 0.2695
  F1          : 0.2512


---
## 3. Bài 3: Mô hình ResNet-18 trên VinaFood21
- Xây dựng từ đầu với các khối **Residual Connection (BasicBlock)**
- Giữa các nhóm Residual Block có một lớp **MaxPooling** (`kernel_size=3, stride=2, padding=0`)
- Huấn luyện với **Adam optimizer**, lưu lại model có **F1-macro** tốt nhất

In [8]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.residual = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        ) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return F.relu(self.residual(x) + self.shortcut(x))

class ResNet18(nn.Module):
    def __init__(self, num_classes=21):
        super().__init__()
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        # 4 nhóm Residual Blocks — xen giữa bởi MaxPool(3, 2, 0) theo yêu cầu đề bài
        self.layer1 = nn.Sequential(BasicBlock(64,  64),  BasicBlock(64,  64))
        self.pool1  = nn.MaxPool2d(kernel_size=3, stride=2, padding=0)

        self.layer2 = nn.Sequential(BasicBlock(64,  128), BasicBlock(128, 128))
        self.pool2  = nn.MaxPool2d(kernel_size=3, stride=2, padding=0)

        self.layer3 = nn.Sequential(BasicBlock(128, 256), BasicBlock(256, 256))
        self.pool3  = nn.MaxPool2d(kernel_size=3, stride=2, padding=0)

        self.layer4 = nn.Sequential(BasicBlock(256, 512), BasicBlock(512, 512))

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        x = self.pool3(self.layer3(x))
        x = self.layer4(x)
        x = self.avgpool(x).view(x.size(0), -1)
        return self.fc(x)

model_resnet18 = ResNet18(num_classes=NUM_CLASSES).to(device)
print(f"ResNet-18 Parameters: {sum(p.numel() for p in model_resnet18.parameters()):,}")

ResNet-18 Parameters: 11,187,285


In [9]:
# Huấn luyện ResNet-18
loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_resnet18.parameters(), lr=1e-3)

EPOCHS  = 10
best_f1_resnet18 = 0.0

for epoch in range(1, EPOCHS + 1):
    model_resnet18.train()
    losses = []
    with tqdm(train_loader, desc=f"[ResNet-18 Epoch {epoch:02d}/{EPOCHS}]", ncols=90) as pb:
        for images, labels in pb:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model_resnet18(images), labels)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            pb.set_postfix(loss=f"{np.mean(losses):.4f}")

    s = evaluate(model_resnet18, test_loader, device)
    print(f"  Test: Acc={s['accuracy']:.4f} | Prec={s['precision']:.4f} | Rec={s['recall']:.4f} | F1={s['f1']:.4f}")

    if s["f1"] > best_f1_resnet18:
        best_f1_resnet18 = s["f1"]
        torch.save(model_resnet18.state_dict(), "./checkpoints/assignment-3/best_model.pth")

print(f"\nResNet-18 - Best Test F1: {best_f1_resnet18:.4f}")

Epoch [1/5] (62.1s) | Loss: 2.8417 | Acc: 0.1733 | Prec: 0.1577 | Rec: 0.1733 | F1: 0.1258
Epoch [2/5] (109.5s) | Loss: 2.4679 | Acc: 0.1876 | Prec: 0.2872 | Rec: 0.1876 | F1: 0.1652
Epoch [3/5] (46.5s) | Loss: 2.3116 | Acc: 0.2333 | Prec: 0.3038 | Rec: 0.2333 | F1: 0.1965
Epoch [4/5] (47.6s) | Loss: 2.1987 | Acc: 0.2752 | Prec: 0.3102 | Rec: 0.2752 | F1: 0.2362
Epoch [5/5] (48.5s) | Loss: 2.0949 | Acc: 0.3152 | Prec: 0.3037 | Rec: 0.3152 | F1: 0.2885

ResNet-18 - Best Test F1: 0.2885


In [10]:
# Đánh giá mô hình ResNet-18 tốt nhất
model_resnet18.load_state_dict(torch.load("./checkpoints/assignment-3/best_model.pth", map_location=device))
res_resnet18 = evaluate(model_resnet18, test_loader, device)

print("=" * 55)
print("     KẾT QUẢ ĐÁNH GIÁ — ResNet-18 trên VinaFood21")
print("=" * 55)
for k, v in res_resnet18.items():
    print(f"  {k.capitalize():<12}: {v:.4f}")
print("=" * 55)

     KẾT QUẢ ĐÁNH GIÁ — ResNet-18 trên VinaFood21
  Accuracy    : 0.3152
  Precision   : 0.3037
  Recall      : 0.3152
  F1          : 0.2885


---
## 4. Bài 4: Fine-tune Pretrained ResNet-50 (HuggingFace)
- Tải mô hình ResNet-50 đã huấn luyện sẵn từ HuggingFace (`microsoft/resnet-50`)
- Thay thế classifier head để phân loại 21 lớp món ăn
- Fine-tune với learning rate nhỏ ($10^{-4}$), optimizer Adam

In [11]:
# Khởi tạo mô hình ResNet-50 từ HuggingFace
model_resnet50 = ResNetForImageClassification.from_pretrained(
    "microsoft/resnet-50",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
).to(device)

print(f"Pretrained ResNet-50 loaded: {sum(p.numel() for p in model_resnet50.parameters() if p.requires_grad):,} parameters")

Pretrained ResNet-50 loaded: 23,549,461 parameters


In [12]:
# Fine-tune ResNet-50
loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_resnet50.parameters(), lr=1e-4)

EPOCHS  = 10
best_f1_resnet50 = 0.0

for epoch in range(1, EPOCHS + 1):
    model_resnet50.train()
    losses = []
    with tqdm(train_loader, desc=f"[ResNet-50 Epoch {epoch:02d}/{EPOCHS}]", ncols=90) as pb:
        for images, labels in pb:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model_resnet50(pixel_values=images).logits
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            pb.set_postfix(loss=f"{np.mean(losses):.4f}")

    s = evaluate(model_resnet50, test_loader, device, is_hf=True)
    print(f"  Test: Acc={s['accuracy']:.4f} | Prec={s['precision']:.4f} | Rec={s['recall']:.4f} | F1={s['f1']:.4f}")

    if s["f1"] > best_f1_resnet50:
        best_f1_resnet50 = s["f1"]
        torch.save(model_resnet50.state_dict(), "./checkpoints/assignment-4/best_model.pth")

print(f"\nResNet-50 - Best Test F1: {best_f1_resnet50:.4f}")

Epoch [1/5] (55.2s) | Loss: 3.0216 | Acc: 0.1505 | Prec: 0.2273 | Rec: 0.1505 | F1: 0.1199
Epoch [2/5] (54.7s) | Loss: 2.9364 | Acc: 0.3067 | Prec: 0.3778 | Rec: 0.3067 | F1: 0.2904
Epoch [3/5] (57.4s) | Loss: 2.8010 | Acc: 0.4295 | Prec: 0.5016 | Rec: 0.4295 | F1: 0.4181
Epoch [4/5] (58.7s) | Loss: 2.5598 | Acc: 0.4943 | Prec: 0.5318 | Rec: 0.4943 | F1: 0.4721
Epoch [5/5] (56.8s) | Loss: 2.1893 | Acc: 0.5552 | Prec: 0.5944 | Rec: 0.5552 | F1: 0.5345

ResNet-50 - Best Test F1: 0.5345


In [13]:
# Đánh giá mô hình ResNet-50 tốt nhất
model_resnet50.load_state_dict(torch.load("./checkpoints/assignment-4/best_model.pth", map_location=device))
res_resnet50 = evaluate(model_resnet50, test_loader, device, is_hf=True)

print("=" * 60)
print("  KẾT QUẢ ĐÁNH GIÁ — ResNet-50 (Pretrained) trên VinaFood21")
print("=" * 60)
for k, v in res_resnet50.items():
    print(f"  {k.capitalize():<12}: {v:.4f}")
print("=" * 60)

  KẾT QUẢ ĐÁNH GIÁ — ResNet-50 (Pretrained) trên VinaFood21
  Accuracy    : 0.5552
  Precision   : 0.5944
  Recall      : 0.5552
  F1          : 0.5345


## 5. Bảng Tổng Hợp So Sánh Kết Quả Cả 3 Mô Hình

In [14]:
print("=" * 68)
print(f"{'Mô hình':<20} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-macro':<10}")
print("=" * 68)

models_res = {
    "GoogLeNet": res_googlenet,
    "ResNet-18": res_resnet18,
    "ResNet-50 (Pretrain)": res_resnet50
}

for name, r in models_res.items():
    print(f"{name:<20} | {r['accuracy']:<10.4f} | {r['precision']:<10.4f} | {r['recall']:<10.4f} | {r['f1']:<10.4f}")

print("=" * 68)

Mô hình                | Accuracy   | Precision  | Recall     | F1-macro  
GoogLeNet              | 0.2695     | 0.3029     | 0.2695     | 0.2512    
ResNet-18              | 0.3152     | 0.3037     | 0.3152     | 0.2885    
ResNet-50 (Pretrained) | 0.5552     | 0.5944     | 0.5552     | 0.5345    
